In [ ]:
# JupyterDash is used because the original CS-340 dashboard
# was designed to run inside Jupyter Notebook.
from jupyter_dash import JupyterDash


# These imports are used to build the dashboard layout,
# interactive components, data table, chart, and map.
import base64
import os
import pandas as pd
import plotly.express as px
import dash_leaflet as dl

from dash import dcc, html, dash_table
from dash.dependencies import Input, Output


# This imports the CRUD module that connects the dashboard
# to the MongoDB AAC animal shelter database.
from animal_shelter import AnimalShelter


# ------------------------------------------------------------
# Database Connection
# ------------------------------------------------------------

# These credentials are used by the AnimalShelter CRUD class
# to connect to MongoDB.
username = "aacuser"
password = "SNHU1234"

# Create the database object.
# The dashboard uses this object to read animal records.
db = AnimalShelter(username, password)


# ------------------------------------------------------------
# Rescue Type Queries
# ------------------------------------------------------------

# These MongoDB queries are used by the radio buttons.
# Each radio button value matches one key in this dictionary.
# This improves the design because the query logic is organized
# in one place instead of being repeated inside the callback.
RESCUE_QUERIES = {
    "Reset": {},

    "Water Rescue": {
        "$and": [
            {"animal_type": "Dog"},
            {
                "breed": {
                    "$in": [
                        "Labrador Retriever Mix",
                        "Chesapeake Bay Retriever",
                        "Newfoundland"
                    ]
                }
            },
            {"sex_upon_outcome": "Intact Male"},
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
        ]
    },

    "Mountain or Wilderness Rescue": {
        "$and": [
            {"animal_type": "Dog"},
            {
                "breed": {
                    "$in": [
                        "German Shepherd",
                        "Alaskan Malamute",
                        "Old English Sheepdog",
                        "Siberian Husky",
                        "Rottweiler"
                    ]
                }
            },
            {"sex_upon_outcome": "Intact Male"},
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
        ]
    },

    "Disaster or Individual Tracking": {
        "$and": [
            {"animal_type": "Dog"},
            {
                "breed": {
                    "$in": [
                        "Doberman Pinscher",
                        "German Shepherd",
                        "Golden Retriever",
                        "Bloodhound",
                        "Rottweiler"
                    ]
                }
            },
            {"sex_upon_outcome": "Intact Male"},
            {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
        ]
    }
}


# ------------------------------------------------------------
# Data Helper Function
# ------------------------------------------------------------

def get_animal_data(query=None):
    """
    Reads animal data from MongoDB and returns it as a cleaned
    pandas dataframe.
    """

    # If no query is passed in, use an empty query.
    # An empty query returns all records.
    query = query or {}

    # Read records from MongoDB using the CRUD read method.
    records = db.read(query)

    # Convert the MongoDB records into a pandas dataframe.
    dataframe = pd.DataFrame.from_records(records)

    # If the query returns no records, return the empty dataframe.
    if dataframe.empty:
        return dataframe

    # MongoDB adds an _id field to each record.
    # Dash DataTable has trouble displaying ObjectId values,
    # so this removes that column if it exists.
    if "_id" in dataframe.columns:
        dataframe.drop(columns=["_id"], inplace=True)

    return dataframe


# Load all animal records when the dashboard first starts.
df = get_animal_data({})


# ------------------------------------------------------------
# Dash App Setup
# ------------------------------------------------------------

# Create the JupyterDash app.
app = JupyterDash(__name__)


# ------------------------------------------------------------
# Logo Setup
# ------------------------------------------------------------

# This uses the logo file from the final submitted dashboard code.
# Make sure this image is in the same folder as the dashboard file.
image_filename = "GraziosoSalvareLogo.png"
encoded_image = None

# Check if the image exists before trying to open it.
# This prevents the dashboard from crashing if the logo file is missing.
if os.path.exists(image_filename):
    with open(image_filename, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()

# Create an empty logo component first.
# If the image loads correctly, this gets replaced with the real logo.
logo_component = html.Div()

if encoded_image:
    logo_component = html.A(
        html.Img(
            src=f"data:image/png;base64,{encoded_image}",
            style={
                "height": "80px",
                "display": "block",
                "marginLeft": "auto",
                "marginRight": "auto"
            }
        ),
        href="https://www.snhu.edu",
        target="_blank"
    )


# ------------------------------------------------------------
# Dashboard Layout
# ------------------------------------------------------------

# The layout controls what the user sees on the dashboard.
# This section was enhanced to make the dashboard cleaner,
# easier to read, and more professional.
app.layout = html.Div(
    style={
        "fontFamily": "Arial, sans-serif",
        "margin": "20px",
        "backgroundColor": "#f7f9fb"
    },
    children=[

        # Header card with logo, title, and unique identifier.
        html.Div(
            style={
                "textAlign": "center",
                "backgroundColor": "white",
                "padding": "20px",
                "borderRadius": "10px",
                "boxShadow": "0 2px 6px rgba(0,0,0,0.1)"
            },
            children=[
                logo_component,
                html.H1("CS-340 Dashboard"),
                html.P("Matthew Tampon")
            ]
        ),

        html.Br(),

        # Filter card.
        # The radio buttons let the user choose which rescue
        # category they want to view.
        html.Div(
            style={
                "backgroundColor": "white",
                "padding": "15px",
                "borderRadius": "10px",
                "boxShadow": "0 2px 6px rgba(0,0,0,0.1)"
            },
            children=[
                html.H3("Rescue Type Filter"),

                dcc.RadioItems(
                    id="filter-type",
                    options=[
                        {"label": "Water Rescue", "value": "Water Rescue"},
                        {
                            "label": "Mountain or Wilderness Rescue",
                            "value": "Mountain or Wilderness Rescue"
                        },
                        {
                            "label": "Disaster or Individual Tracking",
                            "value": "Disaster or Individual Tracking"
                        },
                        {"label": "Reset", "value": "Reset"}
                    ],
                    value="Reset",
                    labelStyle={
                        "display": "inline-block",
                        "margin": "10px"
                    }
                )
            ]
        ),

        html.Br(),

        # Data table card.
        # The table displays the MongoDB animal records.
        # It includes sorting, filtering, paging, and row selection.
        html.Div(
            style={
                "backgroundColor": "white",
                "padding": "15px",
                "borderRadius": "10px",
                "boxShadow": "0 2px 6px rgba(0,0,0,0.1)"
            },
            children=[
                dash_table.DataTable(
                    id="datatable-id",

                    # Create table columns from the dataframe columns.
                    columns=[
                        {
                            "name": column,
                            "id": column,
                            "deletable": False,
                            "selectable": True
                        }
                        for column in df.columns
                    ],

                    # Load the starting table data.
                    data=df.to_dict("records"),

                    # Table interaction settings.
                    editable=False,
                    filter_action="native",
                    sort_action="native",
                    sort_mode="multi",
                    column_selectable="single",
                    row_selectable="single",
                    row_deletable=False,
                    selected_columns=[],
                    selected_rows=[0],

                    # Paging keeps the table easier to read.
                    page_action="native",
                    page_current=0,
                    page_size=10,

                    # Table styling.
                    style_table={"overflowX": "auto"},
                    style_cell={
                        "textAlign": "left",
                        "minWidth": "120px",
                        "width": "150px",
                        "maxWidth": "220px",
                        "whiteSpace": "normal",
                        "fontFamily": "Arial",
                        "fontSize": "13px"
                    },
                    style_header={
                        "backgroundColor": "#e9eef5",
                        "fontWeight": "bold"
                    }
                )
            ]
        ),

        html.Br(),

        # Chart and map section.
        # The chart and map are placed side by side so users can
        # compare the selected data visually.
        html.Div(
            className="row",
            style={
                "display": "flex",
                "gap": "20px",
                "flexWrap": "wrap"
            },
            children=[

                # Pie chart output area.
                html.Div(
                    id="graph-id",
                    style={
                        "flex": "1",
                        "minWidth": "400px",
                        "backgroundColor": "white",
                        "padding": "15px",
                        "borderRadius": "10px",
                        "boxShadow": "0 2px 6px rgba(0,0,0,0.1)"
                    }
                ),

                # Map output area.
                html.Div(
                    id="map-id",
                    style={
                        "flex": "1",
                        "minWidth": "400px",
                        "backgroundColor": "white",
                        "padding": "15px",
                        "borderRadius": "10px",
                        "boxShadow": "0 2px 6px rgba(0,0,0,0.1)"
                    }
                )
            ]
        )
    ]
)


# ------------------------------------------------------------
# Callback: Update Data Table
# ------------------------------------------------------------

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """
    Updates the data table when the user selects a rescue type.
    """

    # Use the selected radio button value to find the matching query.
    query = RESCUE_QUERIES.get(filter_type, {})

    # Read the filtered records from MongoDB.
    filtered_df = get_animal_data(query)

    # Return the filtered data to the dashboard table.
    return filtered_df.to_dict("records")


# ------------------------------------------------------------
# Callback: Update Pie Chart
# ------------------------------------------------------------

@app.callback(
    Output("graph-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("filter-type", "value")
    ]
)
def update_graphs(view_data, filter_type):
    """
    Creates a pie chart from the current data shown in the table.
    """

    # If there is no table data, show a simple message.
    if not view_data:
        return html.Div("No data available for the selected filter.")

    # Convert the current table data into a dataframe.
    filtered_df = pd.DataFrame.from_dict(view_data)

    # The chart depends on the breed column.
    if "breed" not in filtered_df.columns or filtered_df.empty:
        return html.Div("No breed data found.")

    # The reset filter can include a lot of breeds.
    # Limiting it to the top 10 keeps the chart readable.
    if filter_type == "Reset":
        top_breeds = filtered_df["breed"].value_counts().nlargest(10).index
        filtered_df = filtered_df[filtered_df["breed"].isin(top_breeds)]

    # Create the pie chart.
    figure = px.pie(
        filtered_df,
        names="breed",
        title="Preferred Breeds by Filter"
    )

    # Return the chart to the graph output area.
    return [dcc.Graph(figure=figure)]


# ------------------------------------------------------------
# Callback: Highlight Selected Column
# ------------------------------------------------------------

@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """
    Highlights the selected column in the data table.
    """

    return [
        {
            "if": {"column_id": column},
            "background_color": "#D2F3FF"
        }
        for column in selected_columns
    ]


# ------------------------------------------------------------
# Callback: Update Map
# ------------------------------------------------------------

@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(view_data, selected_rows):
    """
    Updates the map marker based on the selected row in the table.
    """

    if not view_data:
        return html.Div("No data available to map.")

    filtered_df = pd.DataFrame.from_dict(view_data)

    if filtered_df.empty:
        return html.Div("No data available to map.")

    if not selected_rows:
        return html.Div("No row selected.")

    row = selected_rows[0]

    if row >= len(filtered_df):
        row = 0

    # Retrieve the selected animal record from the dataframe.
    selected_animal = filtered_df.iloc[row]

    # Verify the required fields exist before building the map.
    required_columns = [
        "location_lat",
        "location_long",
        "breed",
        "name"
    ]

    for column in required_columns:
        if column not in filtered_df.columns:
            return html.Div(f"Missing required column: {column}")

    # Extract location and animal information from the selected record.
    latitude = selected_animal["location_lat"]
    longitude = selected_animal["location_long"]
    breed = selected_animal["breed"]
    name = selected_animal["name"]

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup(
                            [
                                html.H3("Animal Name"),
                                html.P(str(name)),
                                html.H4("Breed"),
                                html.P(str(breed))
                            ]
                        )
                    ]
                )
            ]
        )
    ]

# ------------------------------------------------------------
# Run Dashboard
# ------------------------------------------------------------

# This starts the dashboard server.
app.run_server(debug=True)